In [ ]:
print('Individual OOF accuracies:')
print(f'  LGB : {lgb_oof_acc:.5f}')
print(f'  CAT : {cat_oof_acc:.5f}')
print(f'  XGB : {xgb_oof_acc:.5f}')

# ── Grid-search blend weights over OOF ───────────────────────────────────────
best_acc, best_w = 0, (0.34, 0.33, 0.33)
results = []

for w_lgb in np.arange(0.1, 0.7, 0.05):
    for w_cat in np.arange(0.1, 0.7, 0.05):
        w_xgb = 1.0 - w_lgb - w_cat
        if w_xgb < 0.05:
            continue
        blend = w_lgb * lgb_oof_preds + w_cat * cat_oof_preds + w_xgb * xgb_oof_preds
        acc   = accuracy_score(y, blend.argmax(1))
        results.append((acc, w_lgb, w_cat, w_xgb))
        if acc > best_acc:
            best_acc = acc
            best_w   = (round(w_lgb,2), round(w_cat,2), round(w_xgb,2))

print(f'\nBest blend weights — LGB={best_w[0]}  CAT={best_w[1]}  XGB={best_w[2]}')
print(f'Best ensemble OOF accuracy: {best_acc:.5f}')

# Also try stacked meta-learner (LR on OOF probs) using inner CV
meta_tr = np.hstack([lgb_oof_preds, cat_oof_preds, xgb_oof_preds])
# Evaluate via CV to avoid training-set inflation
from sklearn.model_selection import cross_val_score
meta_cv = cross_val_score(
    LogisticRegression(C=1.0, max_iter=1000, random_state=SEED,
                       multi_class='multinomial', solver='lbfgs'),
    meta_tr, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    scoring='accuracy'
)
print(f'Stacked meta-LR CV accuracy : {meta_cv.mean():.5f} ± {meta_cv.std():.5f}')

## 📊 Cell 10 — Classification report & submission

In [ ]:
# Final blend with best weights
w_lgb, w_cat, w_xgb = best_w

# Final OOF preds for report
final_oof = w_lgb * lgb_oof_preds + w_cat * cat_oof_preds + w_xgb * xgb_oof_preds
print('Classification Report (OOF — proxy for Kaggle score):')
print(classification_report(y, final_oof.argmax(1), target_names=le.classes_))

# ── Generate test predictions ─────────────────────────────────────────────────
final_test = w_lgb * lgb_test_preds + w_cat * cat_test_preds + w_xgb * xgb_test_preds
final_labels = le.inverse_transform(final_test.argmax(1))

submission = pd.DataFrame({'id': test_ids, 'Target': final_labels})
submission.to_csv('solution.csv', index=False)

print('\n✅ Saved solution.csv')
print(f'   Rows        : {len(submission)}')
print(f'   Distribution:\n{submission["Target"].value_counts()}')
submission.head(10)

## 📥 Cell 11 — Download

In [ ]:
from google.colab import files
files.download('solution.csv')
print('✅ Download started → upload to Kaggle!')